# Value and Momentum: From Signals to Factor Forecasts

## Project Objective

The purpose of this project is to empirically explore several forecasting concepts from Grinold and Kahn using a fixed universe of U.S. equities.

The analysis will use monthly observations from 2010 through 2025 and SPY as the market benchmark. Additional historical price data prior to 2010 will be downloaded where necessary to construct signals that require long lookback periods.

Two stock-level signals will be studied:

1. **Momentum**
2. **Price-based Value**

The momentum signal will measure medium-term return continuation using returns from months $t-12$ through $t-2$:

$$
MOM_{i,t}
=
\prod_{k=2}^{12}(1+r_{i,t-k})-1
$$

The most recent month is excluded to avoid mixing medium-term momentum with short-term reversal.

The value signal will use long-term price reversal as a price-only proxy for value. It will be defined as the negative of the stock's cumulative five-year return:

$$
VAL_{i,t}
=
-
\left[
\prod_{k=1}^{60}(1+r_{i,t-k})-1
\right]
$$

A stock that has performed poorly over the previous five years therefore receives a higher value score, while a stock that has performed exceptionally well receives a lower value score.

Each month, both signals will be standardized across the stock universe:

$$
z_{i,t}
=
\frac{g_{i,t}-\bar g_t}{\sigma_{g,t}}
$$

where $g_{i,t}$ is the raw signal for stock $i$ at time $t$.

The project will proceed through four main stages.

### 1. Signal Scaling and Residual Volatility

Stock returns will first be decomposed relative to SPY using

$$
r_{i,t}
=
\alpha_i+\beta_i r_{SPY,t}+\theta_{i,t}.
$$

Residual volatility is

$$
\omega_i
=
Std(\theta_i).
$$

For each signal, we will test whether signal volatility is related to residual return volatility:

$$
Std_{TS}(z_i)
=
a+b\omega_i+\epsilon_i.
$$

This tests whether residual volatility is already embedded in the signal.

### 2. Information Coefficient and Uncertainty

We will test whether each stock-level signal predicts next-month residual returns.

For month $t$:

$$
IC_t
=
Corr_i(z_{i,t},\theta_{i,t+1}).
$$

The resulting time series of ICs will allow us to estimate forecasting skill and study the uncertainty surrounding that estimate.

We will then apply Bayesian shrinkage to avoid treating an uncertain estimated forecasting relationship as if it were known with certainty.

### 3. Value and Momentum Factor Construction

The stock-level signals will be used to construct factor-mimicking portfolios.

Each month, stocks will be sorted by their cross-sectional scores.

For momentum, the highest-scoring tercile will be held long and the lowest-scoring tercile short:

$$
b_{M,t+1}
=
R_{High\ MOM,t+1}
-
R_{Low\ MOM,t+1}.
$$

Similarly, the value factor will be

$$
b_{V,t+1}
=
R_{High\ VAL,t+1}
-
R_{Low\ VAL,t+1}.
$$

These are factor **returns**, whereas $z^M_{i,t}$ and $z^V_{i,t}$ are stock-level **signals**.

We will study the relationship between the two factor returns using both correlation and regression:

$$
b_{M,t}
=
\alpha+\beta b_{V,t}+\epsilon_t.
$$

### 4. Multifactor Forecasting

Finally, the factor returns and forecasting information will be combined using the Grinold and Kahn framework:

$$
E[b_j|g_1]
=
IC\cdot\rho_{j1}\cdot\omega_j\cdot z_1.
$$

This allows information about one factor to provide information about another factor when their returns are correlated.

Stock exposures to the factors can then be estimated using a model of the form

$$
r_{i,t}
=
\alpha_i
+
\beta_{i,MKT}b_{MKT,t}
+
\beta_{i,V}b_{V,t}
+
\beta_{i,M}b_{M,t}
+
\epsilon_{i,t}.
$$

The ultimate goal is to connect:

$$
\text{Prices}
\rightarrow
\text{Signals}
\rightarrow
\text{IC}
\rightarrow
\text{Factor Returns}
\rightarrow
\text{Factor Forecasts}
\rightarrow
\text{Stock Return Forecasts}.
$$

## Backtest Design

The primary study period will be:

$$
2010\text{--}2025.
$$

Because the value signal requires 60 months of historical returns, price data will begin approximately five years before the start of the study.

The stock universe will remain fixed throughout the study. Stocks must therefore have sufficient historical data to construct both signals at the beginning of the analysis.

This introduces survivorship bias because the universe is selected using companies known today. This limitation is accepted because the primary purpose of the project is to study forecasting methodology rather than construct a production-quality historical trading strategy.

SPY will serve as the market benchmark.

## 1. Universe and Price Data

The empirical analysis covers January 2010 through December 2025. However, the price-based value signal requires 60 months of historical returns.

Therefore, price data will be downloaded beginning in 2004. These earlier observations are used only as a **lookback period** for signal construction and are not part of the primary empirical study.

For a stock to enter the fixed universe, it must have sufficient historical price data to construct both signals at the beginning of 2010.

The binding constraint is the value signal:

$$
VAL_{i,t}
=
-\left[
\prod_{k=1}^{60}(1+r_{i,t-k})-1
\right].
$$

Momentum requires a substantially shorter history:

$$
MOM_{i,t}
=
\prod_{k=2}^{12}(1+r_{i,t-k})-1.
$$

Because the universe remains fixed throughout the analysis, stocks without sufficient pre-2010 history will be replaced with companies from the same sector that satisfy the historical-data requirement.

Daily adjusted prices will initially be downloaded. These will subsequently be converted into month-end prices and monthly returns.

In [3]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import statsmodels.api as sm


universe_dict = {
    "Communication Services": ["GOOG", "META", "NFLX"],
    "Consumer Discretionary": ["AMZN", "TSLA", "HD"],
    "Consumer Staples": ["COST", "WMT", "PG"],
    "Energy": ["XOM", "CVX", "COP"],
    "Financials": ["BRK-B", "JPM", "V"],
    "Health Care": ["LLY", "UNH", "JNJ"],
    "Industrials": ["GE", "CAT", "RTX"],
    "Information Technology": ["NVDA", "MSFT", "AAPL"],
    "Materials": ["LIN", "SHW", "APD"],
    "Utilities": ["NEE", "SO", "DUK"],
}

stocks = [
    ticker
    for sector in universe_dict.values()
    for ticker in sector
]

benchmark = "SPY"

len(stocks)

tickers = stocks + [benchmark]

prices = yf.download(
    tickers,
    start="2004-01-01",
    end="2026-01-01",
    auto_adjust=True,
    progress=False
)["Close"]

prices.head()

first_valid_dates = prices.apply(lambda x: x.first_valid_index())

coverage = pd.DataFrame({
    "First Price": first_valid_dates
})

coverage["Eligible"] = coverage["First Price"] <= pd.Timestamp("2005-01-31")

coverage = coverage.sort_values("First Price")

coverage

,First Price,Eligible
Ticker,,
AAPL,2004-01-02,True
UNH,2004-01-02,True
SPY,2004-01-02,True
SO,2004-01-02,True
SHW,2004-01-02,True
RTX,2004-01-02,True
PG,2004-01-02,True
NVDA,2004-01-02,True
NFLX,2004-01-02,True


### Historical Eligibility

A fixed-universe backtest requires more than simply checking whether a stock existed at the beginning of the study.

Because the value signal requires 60 months of historical returns, each stock must have a sufficiently complete price history prior to January 2010.

Missing historical returns will **not** be replaced with zero. A zero return is an economic observation indicating that the stock's price did not change during that period, whereas a missing return indicates that the return is unknown or unavailable.

Replacing missing observations with zero would therefore introduce artificial information into the signal and could distort both cumulative returns and measured volatility.

The original universe contained three stocks without sufficient pre-2010 history:

- META
- TSLA
- V

To maintain three stocks per sector and a fixed 30-stock universe, these will be replaced with older companies from their respective sectors:

- META $\rightarrow$ DIS
- TSLA $\rightarrow$ MCD
- V $\rightarrow$ BAC

The resulting universe will be checked again for sufficient historical coverage before signal construction begins.

In [4]:
universe_dict = {
    "Communication Services": ["GOOG", "DIS", "NFLX"],
    "Consumer Discretionary": ["AMZN", "MCD", "HD"],
    "Consumer Staples": ["COST", "WMT", "PG"],
    "Energy": ["XOM", "CVX", "COP"],
    "Financials": ["BRK-B", "JPM", "BAC"],
    "Health Care": ["LLY", "UNH", "JNJ"],
    "Industrials": ["GE", "CAT", "RTX"],
    "Information Technology": ["NVDA", "MSFT", "AAPL"],
    "Materials": ["LIN", "SHW", "APD"],
    "Utilities": ["NEE", "SO", "DUK"],
}

stocks = [
    ticker
    for sector in universe_dict.values()
    for ticker in sector
]
tickers = stocks + ["SPY"]

prices = yf.download(
    tickers,
    start="2004-01-01",
    end="2026-01-01",
    auto_adjust=True,
    progress=False
)["Close"]

monthly_prices = prices.resample("ME").last()

monthly_prices.head()

monthly_returns = monthly_prices.pct_change(fill_method=None)

monthly_returns.head()

coverage_check = pd.DataFrame({
    "First Monthly Price": monthly_prices.apply(lambda x: x.first_valid_index()),
    "Monthly Prices Before 2010": monthly_prices.loc[: "2009-12-31"].count()
})

coverage_check["Has 60+ Pre-2010 Months"] = (
    coverage_check["Monthly Prices Before 2010"] >= 60
)

coverage_check

,First Monthly Price,Monthly Prices Before 2010,Has 60+ Pre-2010 Months
Ticker,,,
AAPL,2004-01-31,72,True
AMZN,2004-01-31,72,True
APD,2004-01-31,72,True
BAC,2004-01-31,72,True
BRK-B,2004-01-31,72,True
CAT,2004-01-31,72,True
COP,2004-01-31,72,True
COST,2004-01-31,72,True
CVX,2004-01-31,72,True


## 2. Momentum Signal

The first stock-level signal is momentum.

Following the MOM2-12 convention, momentum measures a stock's cumulative return over the previous 12 months while excluding the most recent month.

For a signal formed at the end of month $t$:

$$
MOM_{i,t}
=
\prod_{k=2}^{12}(1+r_{i,t-k})-1
$$

This uses returns from months $t-12$ through $t-2$.

The most recent monthly return, $r_{i,t-1}$, is excluded to separate medium-term momentum from short-term reversal effects.

For example, when constructing a signal at the end of January 2010, the momentum calculation uses historical returns ending before the most recent month. The resulting January signal will then be used to forecast returns in February 2010.

The timing structure throughout the analysis is therefore:

$$
\text{Information available at }t
\rightarrow
\text{Signal at }t
\rightarrow
\text{Return realized at }t+1.
$$

After calculating raw momentum, the signal will be standardized cross-sectionally each month:

$$
z^{MOM}_{i,t}
=
\frac{
MOM_{i,t}-\overline{MOM}_t
}{
\sigma_{MOM,t}
}.
$$

This transforms the raw momentum measure into a relative score.

A positive $z^{MOM}_{i,t}$ indicates that stock $i$ has stronger momentum than the average stock in the universe at month $t$, while a negative score indicates weaker relative momentum.